*   Submitted by: Sudhansh Ranta
*   Roll No.: 41
*   Registeration No.: 12411099
*   Section: D2411 Group:2
*   Course Code: CAB 114
*   Course Name: Model Optimizarion

In [1]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
from sklearn.metrics import classification_report
import os, tempfile, zipfile

print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.19.0


In [2]:
# Load CIFAR-10
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

# Use subset for speed
N_TRAIN = 12000   # you can reduce to 8000 if needed
x_train = x_train[:N_TRAIN]
y_train = y_train[:N_TRAIN]

# Normalize
x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32") / 255.0

# Flatten labels
y_train = y_train.reshape(-1)
y_test  = y_test.reshape(-1)

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

print("Train:", x_train.shape, "Labels:", y_train.shape)
print("Test:", x_test.shape, "Labels:", y_test.shape)


Train: (12000, 32, 32, 3) Labels: (12000,)
Test: (10000, 32, 32, 3) Labels: (10000,)


In [3]:
val_ratio = 0.1
val_size = int(N_TRAIN * val_ratio)

x_val = x_train[:val_size]
y_val = y_train[:val_size]

x_train_small = x_train[val_size:]
y_train_small = y_train[val_size:]

print("Train_small:", x_train_small.shape, "Val:", x_val.shape)


Train_small: (10800, 32, 32, 3) Val: (1200, 32, 32, 3)


In [4]:
def build_model_with_hparams(learning_rate, dense_units):
    model = keras.Sequential([
        keras.layers.Input(shape=(32, 32, 3)),
        keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Flatten(),
        keras.layers.Dense(dense_units, activation="relu"),
        keras.layers.Dense(10, activation="softmax")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

learning_rates = [1e-3, 5e-4]
dense_units_list = [64, 128]

tuning_results = []

print("\n--- Hyperparameter Tuning (1 epoch per combo) ---")
for lr in learning_rates:
    for units in dense_units_list:
        print(f"\nTrying lr={lr}, dense_units={units}")
        model = build_model_with_hparams(lr, units)
        history = model.fit(
            x_train_small, y_train_small,
            batch_size=128,
            epochs=1,                 # only 1 epoch for speed
            validation_data=(x_val, y_val),
            verbose=1
        )
        val_acc = history.history["val_accuracy"][-1]
        tuning_results.append((val_acc, lr, units))
        print(f"Validation accuracy: {val_acc:.4f}")

# pick best combination
tuning_results.sort(reverse=True, key=lambda x: x[0])
best_val_acc, best_lr, best_units = tuning_results[0]

print("\n=== BEST HYPERPARAMETERS FOUND ===")
print(f"Best val accuracy: {best_val_acc:.4f}")
print(f"Best learning rate: {best_lr}")
print(f"Best dense units: {best_units}")



--- Hyperparameter Tuning (1 epoch per combo) ---

Trying lr=0.001, dense_units=64
85/85 ━━━━━━━━━━━━━━━━━━━━ 23s 216ms/step - accuracy: 0.2163 - loss: 2.1299 - val_accuracy: 0.4100 - val_loss: 1.6627
Validation accuracy: 0.4100

Trying lr=0.001, dense_units=128
85/85 ━━━━━━━━━━━━━━━━━━━━ 33s 315ms/step - accuracy: 0.2363 - loss: 2.0799 - val_accuracy: 0.4267 - val_loss: 1.5817
Validation accuracy: 0.4267

Trying lr=0.0005, dense_units=64
85/85 ━━━━━━━━━━━━━━━━━━━━ 19s 190ms/step - accuracy: 0.1976 - loss: 2.1827 - val_accuracy: 0.3475 - val_loss: 1.8051
Validation accuracy: 0.3475

Trying lr=0.0005, dense_units=128
85/85 ━━━━━━━━━━━━━━━━━━━━ 19s 206ms/step - accuracy: 0.2238 - loss: 2.1290 - val_accuracy: 0.3967 - val_loss: 1.6819
Validation accuracy: 0.3967

=== BEST HYPERPARAMETERS FOUND ===
Best val accuracy: 0.4267
Best learning rate: 0.001
Best dense units: 128


In [5]:
def build_baseline_model(learning_rate, dense_units):
    model = keras.Sequential([
        keras.layers.Input(shape=(32, 32, 3)),
        keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Flatten(),
        keras.layers.Dense(dense_units, activation="relu"),
        keras.layers.Dense(10, activation="softmax")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

baseline_model = build_baseline_model(best_lr, best_units)
baseline_model.summary()


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_8 (Conv2D)               │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 128)            │       524,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 545,098 (2.08 MB)

 Trainable params: 545,098 (2.08 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
batch_size = 128
epochs = 3

print("\n--- Training Final Baseline Model ---")
baseline_history = baseline_model.fit(
    x_train, y_train,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=0.1,
    verbose=1
)



--- Training Final Baseline Model ---
Epoch 1/3
85/85 ━━━━━━━━━━━━━━━━━━━━ 18s 195ms/step - accuracy: 0.2256 - loss: 2.0920 - val_accuracy: 0.4200 - val_loss: 1.5964
Epoch 2/3
85/85 ━━━━━━━━━━━━━━━━━━━━ 16s 193ms/step - accuracy: 0.4582 - loss: 1.5470 - val_accuracy: 0.4742 - val_loss: 1.4344
Epoch 3/3
85/85 ━━━━━━━━━━━━━━━━━━━━ 18s 213ms/step - accuracy: 0.5088 - loss: 1.3948 - val_accuracy: 0.5208 - val_loss: 1.3174


In [7]:
print("\n--- Baseline Evaluation ---")
baseline_loss, baseline_acc = baseline_model.evaluate(x_test, y_test, verbose=0)
print(f"Baseline Test Accuracy: {baseline_acc * 100:.2f}%")

y_pred_baseline = np.argmax(baseline_model.predict(x_test, verbose=0), axis=1)

print("\n--- Baseline Classification Report ---")
print(classification_report(y_test, y_pred_baseline, target_names=class_names))



--- Baseline Evaluation ---
Baseline Test Accuracy: 53.25%

--- Baseline Classification Report ---
              precision    recall  f1-score   support

    airplane       0.57      0.60      0.59      1000
  automobile       0.76      0.49      0.60      1000
        bird       0.48      0.24      0.32      1000
         cat       0.38      0.42      0.40      1000
        deer       0.57      0.29      0.39      1000
         dog       0.40      0.58      0.47      1000
        frog       0.57      0.66      0.61      1000
       horse       0.56      0.61      0.59      1000
        ship       0.64      0.71      0.67      1000
       truck       0.53      0.70      0.60      1000

    accuracy                           0.53     10000
   macro avg       0.55      0.53      0.52     10000
weighted avg       0.55      0.53      0.52     10000



In [8]:
def create_pruned_copy(model, pruning_fraction=0.5):
    """
    Create a new model where a given fraction of the smallest-magnitude
    weights are set to zero (manual pruning).
    """
    # Clone architecture
    pruned_model = keras.models.clone_model(model)
    pruned_model.build(model.input_shape)
    pruned_model.set_weights(model.get_weights())

    # Get all weights
    weights = pruned_model.get_weights()

    # Concatenate all non-bias weights to find global threshold
    all_weights = []
    for w in weights:
        if w.ndim > 1:  # skip biases (1D)
            all_weights.append(w.flatten())
    all_weights = np.concatenate(all_weights)

    # Determine threshold by pruning_fraction
    k = int(pruning_fraction * all_weights.size)
    if k <= 0:
        return pruned_model  # nothing to prune
    threshold = np.partition(np.abs(all_weights), k)[k]

    # Zero out weights below threshold
    new_weights = []
    for w in weights:
        if w.ndim > 1:
            w = np.where(np.abs(w) < threshold, 0.0, w)
        new_weights.append(w)

    pruned_model.set_weights(new_weights)

    # Compile pruned model (same optimizer/loss/metrics)
    pruned_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=best_lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return pruned_model

print("\n--- Creating manually pruned model (50% smallest weights -> 0) ---")
pruned_model = create_pruned_copy(baseline_model, pruning_fraction=0.5)
pruned_model.summary()



--- Creating manually pruned model (50% smallest weights -> 0) ---


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_8 (Conv2D)               │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 128)            │       524,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 545,098 (2.08 MB)

 Trainable params: 545,098 (2.08 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
print("\n--- Fine-tuning Pruned Model ---")
pruned_epochs = 2

pruned_history = pruned_model.fit(
    x_train, y_train,
    batch_size=batch_size,
    epochs=pruned_epochs,
    validation_split=0.1,
    verbose=1
)



--- Fine-tuning Pruned Model ---
Epoch 1/2
85/85 ━━━━━━━━━━━━━━━━━━━━ 18s 193ms/step - accuracy: 0.5445 - loss: 1.3104 - val_accuracy: 0.5258 - val_loss: 1.3028
Epoch 2/2
85/85 ━━━━━━━━━━━━━━━━━━━━ 20s 190ms/step - accuracy: 0.6127 - loss: 1.1315 - val_accuracy: 0.5500 - val_loss: 1.2406


In [10]:
print("\n--- Pruned Model Evaluation ---")
pruned_loss, pruned_acc = pruned_model.evaluate(x_test, y_test, verbose=0)
print(f"Pruned Test Accuracy: {pruned_acc * 100:.2f}%")

y_pred_pruned = np.argmax(pruned_model.predict(x_test, verbose=0), axis=1)

print("\n--- Pruned Model Classification Report ---")
print(classification_report(y_test, y_pred_pruned, target_names=class_names))



--- Pruned Model Evaluation ---
Pruned Test Accuracy: 56.44%

--- Pruned Model Classification Report ---
              precision    recall  f1-score   support

    airplane       0.59      0.69      0.63      1000
  automobile       0.64      0.79      0.71      1000
        bird       0.39      0.52      0.44      1000
         cat       0.41      0.42      0.41      1000
        deer       0.52      0.43      0.47      1000
         dog       0.48      0.52      0.50      1000
        frog       0.82      0.45      0.58      1000
       horse       0.59      0.66      0.62      1000
        ship       0.69      0.65      0.67      1000
       truck       0.74      0.52      0.61      1000

    accuracy                           0.56     10000
   macro avg       0.59      0.56      0.56     10000
weighted avg       0.59      0.56      0.56     10000



In [11]:
def get_model_size(filepath):
    return os.path.getsize(filepath)

# Save baseline
baseline_path = tempfile.mkstemp(".h5")[1]
baseline_model.save(baseline_path, include_optimizer=False)
baseline_size = get_model_size(baseline_path)

# Save pruned
pruned_path = tempfile.mkstemp(".h5")[1]
pruned_model.save(pruned_path, include_optimizer=False)
pruned_size = get_model_size(pruned_path)

print("\n--- Model Size Comparison (raw .h5 size) ---")
print(f"Baseline model size:   {baseline_size / 1024:.2f} KB")
print(f"Pruned model size:     {pruned_size / 1024:.2f} KB")
print(f"Size reduction:        {(1 - pruned_size / baseline_size) * 100:.2f}%")



--- Model Size Comparison (raw .h5 size) ---
Baseline model size:   2159.23 KB
Pruned model size:     2159.23 KB
Size reduction:        0.00%


In [12]:
print("\n" + "="*45)
print("FINAL COMPARISON SUMMARY")
print("="*45)
print(f"Best hyperparams -> lr={best_lr}, dense_units={best_units}")
print(f"Baseline Accuracy:   {baseline_acc * 100:.2f}%")
print(f"Pruned Accuracy:     {pruned_acc * 100:.2f}%")
print("-"*45)
print(f"Baseline Size:       {baseline_size / 1024:.2f} KB")
print(f"Pruned Size:         {pruned_size / 1024:.2f} KB")
print(f"Size Reduction:      {(1 - pruned_size / baseline_size) * 100:.2f}%")
print("="*45)



FINAL COMPARISON SUMMARY
Best hyperparams -> lr=0.001, dense_units=128
Baseline Accuracy:   53.25%
Pruned Accuracy:     56.44%
---------------------------------------------
Baseline Size:       2159.23 KB
Pruned Size:         2159.23 KB
Size Reduction:      0.00%
